# 02 - Représentation : règle graphique, signatures, style SLD

**Objectif**
- Écrire une règle graphique (`GRAPHIC`) construite sur une VIEW, et les signatures (`SIGN`) qu'elle utilise, dans un transfert XTF.
- Convertir l'ensemble en style SLD avec `interlis convert-sld`.
- Lire le SLD obtenu et vérifier qu'il reflète les règles écrites.

Cas d'étude : les distances par rapport à la forêt (MGDM Waldabstandslinien), dont la VIEW `linien` est fournie par le dépôt.

## Les trois éléments d'une représentation INTERLIS

| Élément | Où il est écrit | Rôle |
|---|---|---|
| VIEW | modèle dérivé (notebook 01) | les attributs et la géométrie à dessiner |
| règle graphique `GRAPHIC` | modèle de représentation (`.ili`) | quelle signature pour quels objets, avec quelle géométrie et quelle priorité |
| signature `SIGN` | transfert de données (`.xtf`, modèle `StandardSymbology`) | couleurs, styles de ligne, symboles, textes |

Les règles graphiques sont du modèle ; les signatures sont des données. Changer une couleur ne demande donc pas de changer le modèle.

In [ ]:
!mkdir -p modeles sorties

## Étape 1 - Lire une représentation existante

La collection `waldabstandslinien` du dépôt contient un modèle de représentation et ses signatures.

In [ ]:
!cat ../collections/waldabstandslinien/symbology.ili

Lecture :

- `SIGN BASKET Signs ~ StandardSymbology.StandardSigns` annonce que les signatures sont dans un transfert XTF, sous le thème `StandardSigns` ; `OBJECTS OF PolylineSign: WaldGreen` nomme celles que le modèle utilise.
- `GRAPHIC Waldabstand_Graphics BASED ON ...Views.linien` rattache la règle graphique à la VIEW `linien`.
- `WaldabstandRule` est une règle de type `PolylineSign` : pour chaque ligne de la VIEW, dessiner la signature `WaldGreen` (`Sign`), avec la géométrie `geometrie`, à la priorité 100 (`Priority`).

Le fichier de signatures contient trois objets, liés entre eux par `ili:ref` :

In [ ]:
!sed -n '/<StandardSigns/,/<\/StandardSigns>/p' ../collections/waldabstandslinien/signs.xtf

- `Color` : une couleur en CIE LCH (`L` clarté, `C` chroma, `H` teinte, `T` opacité) ;
- `LineStyle_Solid` : un trait plein, qui référence la couleur ;
- `PolylineSign` : la signature nommée `WaldGreen`, qui référence le style de ligne. C'est ce nom que la règle graphique cite.

## Étape 2 - Convertir en SLD

`fetch-models` télécharge `StandardSymbology` et le modèle de base ; `convert-sld` lit le modèle de représentation, résout les signatures dans le XTF (`--sign-xtf`) et écrit un fichier SLD par `GRAPHIC`.

In [ ]:
!interlis fetch-models ../collections/waldabstandslinien/views.ili ../collections/waldabstandslinien/symbology.ili --repo ../collections/waldabstandslinien -o modeles

In [ ]:
!interlis convert-sld ../collections/waldabstandslinien/symbology.ili --repo modeles --repo ../collections/waldabstandslinien \
    --sign-xtf ../collections/waldabstandslinien/signs.xtf --graphic Waldabstand_Graphics -o sorties/depot.sld
!cat sorties/depot.sld

Le SLD contient une règle par règle graphique, avec le type de géométrie dessiné (`LineSymbolizer`) et la couleur convertie en sRGB (`#00ff00` pour la couleur LCH ci-dessus).

## Étape 3 - Écrire sa propre représentation

Le modèle `MaSymbologie` ci-dessous reprend la même VIEW, avec une règle graphique qui sélectionne les lignes en vigueur (`rechtsstatus == #inKraft`) par un `WHERE`. Les signatures sont dans un XTF écrit à part : deux couleurs (vert et rouge) avec leur trait plein et leur signature, `LigneVerte` et `LigneRouge`.

In [ ]:
%%writefile sorties/MaSymbologie.ili
INTERLIS 2.4;

MODEL MaSymbologie (fr)
  AT "mailto:exercice@example.org"
  VERSION "2025-03-27" =

  IMPORTS StandardSymbology;
  IMPORTS Waldabstandslinien_Views;

  SIGN BASKET Signs ~ StandardSymbology.StandardSigns
    OBJECTS OF PolylineSign: LigneVerte, LigneRouge;

  TOPIC Graphics =
    DEPENDS ON Waldabstandslinien_Views.Views;

    GRAPHIC Lignes_Graphics
      BASED ON Waldabstandslinien_Views.Views.linien =
      EnVigueurRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #inKraft) (
          Sign := {LigneVerte};
          Geometry := geometrie;
          Priority := 100);

    END Lignes_Graphics;

  END Graphics;

END MaSymbologie.

In [ ]:
%%writefile sorties/signes.xtf
<?xml version="1.0" encoding="UTF-8"?>
<ili:transfer xmlns:ili="http://www.interlis.ch/xtf/2.4/INTERLIS"
  xmlns="http://www.interlis.ch/xtf/2.4/StandardSymbology"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <ili:headersection>
    <ili:models>
      <ili:model>MaSymbologie</ili:model>
    </ili:models>
    <ili:sender>notebook-02</ili:sender>
  </ili:headersection>
  <ili:datasection>
    <StandardSigns ili:bid="Signs00000001">
      <Color ili:tid="1">
        <Name>vert</Name>
        <L>87.7</L>
        <C>119.8</C>
        <H>136.02</H>
        <T>1.0</T>
      </Color>
      <Color ili:tid="2">
        <Name>rouge</Name>
        <L>53.2</L>
        <C>104.6</C>
        <H>40.0</H>
        <T>1.0</T>
      </Color>
      <LineStyle_Solid ili:tid="41">
        <Name>Vert_Solid</Name>
        <Color ili:ref="1"></Color>
      </LineStyle_Solid>
      <LineStyle_Solid ili:tid="42">
        <Name>Rouge_Solid</Name>
        <Color ili:ref="2"></Color>
      </LineStyle_Solid>
      <PolylineSign ili:tid="101">
        <ili:Name>LigneVerte</ili:Name>
        <Style ili:ref="41">
          <PolylineSignLineStyleAssoc>
            <Offset>0.0</Offset>
          </PolylineSignLineStyleAssoc>
        </Style>
      </PolylineSign>
      <PolylineSign ili:tid="102">
        <ili:Name>LigneRouge</ili:Name>
        <Style ili:ref="42">
          <PolylineSignLineStyleAssoc>
            <Offset>0.0</Offset>
          </PolylineSignLineStyleAssoc>
        </Style>
      </PolylineSign>
    </StandardSigns>
  </ili:datasection>
</ili:transfer>

In [ ]:
!interlis convert-sld sorties/MaSymbologie.ili --repo modeles --repo ../collections/waldabstandslinien \
    --sign-xtf sorties/signes.xtf -o sorties/ma.sld
!grep -E "Rule>|<ogc:PropertyName|<ogc:Literal|stroke\"" sorties/ma.sld

La couleur LCH `87.7 / 119.8 / 136.02` donne bien `#00ff00` : la conversion de l'espace colorimétrique est faite par ili2ogc.

## À vous

Les deux exercices complètent cette représentation : une deuxième règle graphique avec la signature `LigneRouge` déjà définie, puis une nouvelle signature avec sa règle.

### À vous : une règle graphique pour les lignes modifiées

Dans la cellule suivante, ajouter au modèle une deuxième règle graphique, `ModifieeRule`, qui dessine avec la signature `LigneRouge` les lignes dont `rechtsstatus` vaut `#AenderungMitVorwirkung`. S'inspirer de `EnVigueurRule` ; la priorité doit être supérieure (101), pour que la règle soit dessinée après. Remplacer les `...`, puis lancer la cellule de commande qui suit.

In [ ]:
%%writefile sorties/MaSymbologie.ili
INTERLIS 2.4;

MODEL MaSymbologie (fr)
  AT "mailto:exercice@example.org"
  VERSION "2025-03-27" =

  IMPORTS StandardSymbology;
  IMPORTS Waldabstandslinien_Views;

  SIGN BASKET Signs ~ StandardSymbology.StandardSigns
    OBJECTS OF PolylineSign: LigneVerte, LigneRouge;

  TOPIC Graphics =
    DEPENDS ON Waldabstandslinien_Views.Views;

    GRAPHIC Lignes_Graphics
      BASED ON Waldabstandslinien_Views.Views.linien =
      EnVigueurRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #inKraft) (
          Sign := {LigneVerte};
          Geometry := geometrie;
          Priority := 100);

      ModifieeRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (...) (
          Sign := {...};
          Geometry := ...;
          Priority := ...);

    END Lignes_Graphics;

  END Graphics;

END MaSymbologie.

In [ ]:
!interlis convert-sld sorties/MaSymbologie.ili --repo modeles --repo ../collections/waldabstandslinien \
    --sign-xtf sorties/signes.xtf -o sorties/ma2.sld
!grep -E "Rule>|<ogc:PropertyName|<ogc:Literal|stroke\"" sorties/ma2.sld

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
sld = open("sorties/ma2.sld").read()
assert sld.count("<se:Rule>") == 2, f"2 règles attendues dans le SLD, {sld.count('<se:Rule>')} trouvée(s)"
assert "AenderungMitVorwirkung" in sld, "le filtre sur AenderungMitVorwirkung manque"
assert "#ff0000" in sld, "la couleur rouge (#ff0000) manque"
print("Exercice réussi.")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
%%writefile sorties/MaSymbologie.ili
INTERLIS 2.4;

MODEL MaSymbologie (fr)
  AT "mailto:exercice@example.org"
  VERSION "2025-03-27" =

  IMPORTS StandardSymbology;
  IMPORTS Waldabstandslinien_Views;

  SIGN BASKET Signs ~ StandardSymbology.StandardSigns
    OBJECTS OF PolylineSign: LigneVerte, LigneRouge;

  TOPIC Graphics =
    DEPENDS ON Waldabstandslinien_Views.Views;

    GRAPHIC Lignes_Graphics
      BASED ON Waldabstandslinien_Views.Views.linien =
      EnVigueurRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #inKraft) (
          Sign := {LigneVerte};
          Geometry := geometrie;
          Priority := 100);

      ModifieeRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #AenderungMitVorwirkung) (
          Sign := {LigneRouge};
          Geometry := geometrie;
          Priority := 101);

    END Lignes_Graphics;

  END Graphics;

END MaSymbologie.

## Priorité et ordre de dessin

`Priority` fixe la passe de dessin : une règle de priorité plus élevée est dessinée après, donc au-dessus. Dans le SLD, les règles sont écrites dans cet ordre. Les moteurs de rendu n'appliquent pas tous cet ordre ; c'est une limite du moteur, pas du modèle.

### À vous : une nouvelle signature, avec sa règle

Ajouter une troisième catégorie, en orange, pour `rechtsstatus` égal à `#AenderungOhneVorwirkung`. Trois éléments à compléter, dans deux cellules :

1. dans `signes.xtf`, à la place du commentaire : couleur, trait plein et signature `LigneOrange`. Reprendre le schéma des signatures existantes, avec des `tid` encore libres (3, 43, 103) et la couleur LCH `L=67.8`, `C=87.6`, `H=58.7`, `T=1.0` ;
2. dans `MaSymbologie.ili` : déclarer `LigneOrange` dans le `SIGN BASKET` (après `LigneRouge`) ;
3. dans le même fichier : la règle graphique `SansVorwirkungRule`, priorité 102.

In [ ]:
%%writefile sorties/signes.xtf
<?xml version="1.0" encoding="UTF-8"?>
<ili:transfer xmlns:ili="http://www.interlis.ch/xtf/2.4/INTERLIS"
  xmlns="http://www.interlis.ch/xtf/2.4/StandardSymbology"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <ili:headersection>
    <ili:models>
      <ili:model>MaSymbologie</ili:model>
    </ili:models>
    <ili:sender>notebook-02</ili:sender>
  </ili:headersection>
  <ili:datasection>
    <StandardSigns ili:bid="Signs00000001">
      <Color ili:tid="1">
        <Name>vert</Name>
        <L>87.7</L>
        <C>119.8</C>
        <H>136.02</H>
        <T>1.0</T>
      </Color>
      <Color ili:tid="2">
        <Name>rouge</Name>
        <L>53.2</L>
        <C>104.6</C>
        <H>40.0</H>
        <T>1.0</T>
      </Color>
      <LineStyle_Solid ili:tid="41">
        <Name>Vert_Solid</Name>
        <Color ili:ref="1"></Color>
      </LineStyle_Solid>
      <LineStyle_Solid ili:tid="42">
        <Name>Rouge_Solid</Name>
        <Color ili:ref="2"></Color>
      </LineStyle_Solid>
      <PolylineSign ili:tid="101">
        <ili:Name>LigneVerte</ili:Name>
        <Style ili:ref="41">
          <PolylineSignLineStyleAssoc>
            <Offset>0.0</Offset>
          </PolylineSignLineStyleAssoc>
        </Style>
      </PolylineSign>
      <PolylineSign ili:tid="102">
        <ili:Name>LigneRouge</ili:Name>
        <Style ili:ref="42">
          <PolylineSignLineStyleAssoc>
            <Offset>0.0</Offset>
          </PolylineSignLineStyleAssoc>
        </Style>
      </PolylineSign>
      <!-- Color (tid 3), LineStyle_Solid (tid 43) et PolylineSign (tid 103, LigneOrange) -->
    </StandardSigns>
  </ili:datasection>
</ili:transfer>

In [ ]:
%%writefile sorties/MaSymbologie.ili
INTERLIS 2.4;

MODEL MaSymbologie (fr)
  AT "mailto:exercice@example.org"
  VERSION "2025-03-27" =

  IMPORTS StandardSymbology;
  IMPORTS Waldabstandslinien_Views;

  SIGN BASKET Signs ~ StandardSymbology.StandardSigns
    OBJECTS OF PolylineSign: LigneVerte, LigneRouge;

  TOPIC Graphics =
    DEPENDS ON Waldabstandslinien_Views.Views;

    GRAPHIC Lignes_Graphics
      BASED ON Waldabstandslinien_Views.Views.linien =
      EnVigueurRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #inKraft) (
          Sign := {LigneVerte};
          Geometry := geometrie;
          Priority := 100);

      ModifieeRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #AenderungMitVorwirkung) (
          Sign := {LigneRouge};
          Geometry := geometrie;
          Priority := 101);

      SansVorwirkungRule OF ...

    END Lignes_Graphics;

  END Graphics;

END MaSymbologie.

In [ ]:
!interlis convert-sld sorties/MaSymbologie.ili --repo modeles --repo ../collections/waldabstandslinien \
    --sign-xtf sorties/signes.xtf -o sorties/ma3.sld
!grep -E "Rule>|<ogc:PropertyName|<ogc:Literal|stroke\"" sorties/ma3.sld

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
sld = open("sorties/ma3.sld").read()
assert sld.count("<se:Rule>") == 3, f"3 règles attendues dans le SLD, {sld.count('<se:Rule>')} trouvée(s)"
assert "AenderungOhneVorwirkung" in sld, "le filtre sur AenderungOhneVorwirkung manque"
couleur = sld.split('name="stroke">#')[3][:6]
rouge, vert, bleu = (int(couleur[i:i + 2], 16) for i in (0, 2, 4))
assert rouge > 200 and 90 < vert < 190 and bleu < 60, f"la couleur #{couleur} n'est pas orange"
print("Exercice réussi : la troisième règle est orange (#" + couleur + ").")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
%%writefile sorties/signes.xtf
<?xml version="1.0" encoding="UTF-8"?>
<ili:transfer xmlns:ili="http://www.interlis.ch/xtf/2.4/INTERLIS"
  xmlns="http://www.interlis.ch/xtf/2.4/StandardSymbology"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <ili:headersection>
    <ili:models>
      <ili:model>MaSymbologie</ili:model>
    </ili:models>
    <ili:sender>notebook-02</ili:sender>
  </ili:headersection>
  <ili:datasection>
    <StandardSigns ili:bid="Signs00000001">
      <Color ili:tid="1">
        <Name>vert</Name>
        <L>87.7</L>
        <C>119.8</C>
        <H>136.02</H>
        <T>1.0</T>
      </Color>
      <Color ili:tid="2">
        <Name>rouge</Name>
        <L>53.2</L>
        <C>104.6</C>
        <H>40.0</H>
        <T>1.0</T>
      </Color>
      <LineStyle_Solid ili:tid="41">
        <Name>Vert_Solid</Name>
        <Color ili:ref="1"></Color>
      </LineStyle_Solid>
      <LineStyle_Solid ili:tid="42">
        <Name>Rouge_Solid</Name>
        <Color ili:ref="2"></Color>
      </LineStyle_Solid>
      <PolylineSign ili:tid="101">
        <ili:Name>LigneVerte</ili:Name>
        <Style ili:ref="41">
          <PolylineSignLineStyleAssoc>
            <Offset>0.0</Offset>
          </PolylineSignLineStyleAssoc>
        </Style>
      </PolylineSign>
      <PolylineSign ili:tid="102">
        <ili:Name>LigneRouge</ili:Name>
        <Style ili:ref="42">
          <PolylineSignLineStyleAssoc>
            <Offset>0.0</Offset>
          </PolylineSignLineStyleAssoc>
        </Style>
      </PolylineSign>
      <Color ili:tid="3">
        <Name>orange</Name>
        <L>67.8</L>
        <C>87.6</C>
        <H>58.7</H>
        <T>1.0</T>
      </Color>
      <LineStyle_Solid ili:tid="43">
        <Name>Orange_Solid</Name>
        <Color ili:ref="3"></Color>
      </LineStyle_Solid>
      <PolylineSign ili:tid="103">
        <ili:Name>LigneOrange</ili:Name>
        <Style ili:ref="43">
          <PolylineSignLineStyleAssoc>
            <Offset>0.0</Offset>
          </PolylineSignLineStyleAssoc>
        </Style>
      </PolylineSign>
    </StandardSigns>
  </ili:datasection>
</ili:transfer>

In [ ]:
%%writefile sorties/MaSymbologie.ili
INTERLIS 2.4;

MODEL MaSymbologie (fr)
  AT "mailto:exercice@example.org"
  VERSION "2025-03-27" =

  IMPORTS StandardSymbology;
  IMPORTS Waldabstandslinien_Views;

  SIGN BASKET Signs ~ StandardSymbology.StandardSigns
    OBJECTS OF PolylineSign: LigneVerte, LigneRouge, LigneOrange;

  TOPIC Graphics =
    DEPENDS ON Waldabstandslinien_Views.Views;

    GRAPHIC Lignes_Graphics
      BASED ON Waldabstandslinien_Views.Views.linien =
      EnVigueurRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #inKraft) (
          Sign := {LigneVerte};
          Geometry := geometrie;
          Priority := 100);

      ModifieeRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #AenderungMitVorwirkung) (
          Sign := {LigneRouge};
          Geometry := geometrie;
          Priority := 101);

      SansVorwirkungRule OF StandardSymbology.StandardSigns.PolylineSign:
        WHERE (rechtsstatus == #AenderungOhneVorwirkung) (
          Sign := {LigneOrange};
          Geometry := geometrie;
          Priority := 102);

    END Lignes_Graphics;

  END Graphics;

END MaSymbologie.

## Utiliser le SLD

Le fichier `ma3.sld` est un style OGC standard. Il se charge dans QGIS (propriétés de la couche, onglet Symbologie, charger le style), dans GeoServer, MapServer ou QGIS Server. Dans la pile Docker du dépôt, le chargeur (`loader/load.py`) lance `convert-sld` pour chaque `GRAPHIC` et pygeoapi dessine la carte (OGC API - Maps) avec ces styles ; voir le notebook 04.

Ce que le SLD ne dit pas : la couche à laquelle il s'applique. Le nom de la VIEW (`linien`) figure dans `FeatureTypeName` ; la table ou la vue du même nom doit exister dans la source de données.

## À retenir

- La règle graphique est écrite dans un modèle, sur une VIEW ; les signatures sont des données XTF.
- `convert-sld` résout les signatures citées dans `Sign := {...}` à partir du XTF donné par `--sign-xtf`.
- `WHERE` sur un attribut de la VIEW donne un filtre dans le SLD ; `Priority` donne l'ordre des règles.
- Les couleurs LCH sont converties en sRGB.

## Suite

Notebook `03_ili2ogc.ipynb` : la chaîne complète de commandes sur un transfert réel du dépôt.